In [11]:
import pandas as pd
import numpy as np

In [12]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_DTU, Delhi - CPCB.xlsx",skiprows=16)

In [13]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,...,Benzene,Toluene,Eth-Benzene,MP-Xylene,RH,WS,WD,BP,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,70.18,274.11,32.80,38.76,71.48,55.30,65.05,1.02,...,0.66,NaN,1.42,0.07,88.96,0.92,260.03,NaN,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,82.18,332.42,28.72,37.65,66.36,51.83,64.96,1.21,...,0.71,NaN,1.31,0.05,90.72,0.84,287.60,NaN,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,112.12,390.26,45.75,45.38,90.43,73.86,65.22,2.06,...,0.72,NaN,1.74,0.14,93.56,0.88,273.52,NaN,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,108.82,380.46,53.07,55.13,107.21,75.13,65.52,2.57,...,0.76,NaN,1.50,0.06,93.55,1.83,113.95,NaN,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,81.60,321.22,36.98,52.94,89.94,56.82,66.29,1.67,...,NaN,NaN,NaN,NaN,85.92,1.33,141.80,NaN,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,160.58,476.06,22.24,52.32,45.89,45.75,24.63,1.57,...,0.48,NaN,0.52,0.14,66.98,0.67,287.70,NaN,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,146.02,442.23,17.85,56.66,44.67,44.32,26.13,1.62,...,0.49,NaN,0.52,0.13,69.54,0.37,286.18,NaN,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,216.05,422.28,20.53,48.60,42.60,44.16,26.51,1.43,...,0.47,NaN,0.54,0.15,68.87,0.54,299.04,NaN,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,239.45,384.87,20.73,49.18,42.96,36.22,25.90,1.30,...,0.49,NaN,0.51,0.14,67.72,0.71,300.93,NaN,0.0,0.0


In [14]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 20)


In [15]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Toluene']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date      0
To Date        0
PM2.5          0
PM10           0
NO             0
NO2            0
NOx            0
NH3            0
SO2            0
CO             0
Ozone          0
Benzene        0
Eth-Benzene    0
MP-Xylene      0
RH             0
WS             0
WD             0
RF             0
TOT-RF         0
dtype: int64


In [16]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")

# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")

In [ ]:
# ---------- 5. Convert date columns to datetime ----------
if 'From Date' in df.columns:
    df['From Date'] = pd.to_datetime(df['From Date'], errors='coerce')
if 'To Date' in df.columns:
    df['To Date'] = pd.to_datetime(df['To Date'], errors='coerce')

In [17]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 19)
          From Date           To Date   PM2.5    PM10     NO    NO2   NOx  \
0  01-01-2025 00:00  02-01-2025 00:00   70.18  274.11  32.80  38.76  35.8   
1  02-01-2025 00:00  03-01-2025 00:00   82.18  332.42  28.72  37.65  35.8   
2  03-01-2025 00:00  04-01-2025 00:00  112.12  390.26  24.67  45.38  35.8   
3  04-01-2025 00:00  05-01-2025 00:00  108.82  380.46  24.67  55.13  35.8   
4  05-01-2025 00:00  06-01-2025 00:00   81.60  321.22  36.98  52.94  35.8   

     NH3    SO2    CO  Ozone  Benzene  Eth-Benzene  MP-Xylene     RH    WS  \
0  55.30  65.05  1.02  38.87     0.66         0.62       0.07  88.96  0.92   
1  51.83  64.96  1.21  39.08     0.71         0.62       0.19  90.72  0.84   
2  47.82  65.22  0.63  43.49     0.72         0.62       0.14  93.56  0.88   
3  47.82  65.52  0.63  37.96     0.76         0.62       0.06  93.55  1.83   
4  56.82  66.29  0.63  37.81     0.89         0.62       0.19  85.92  1.33   

       WD   RF  TOT-RF  
0  260.03  0.0     0

In [18]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [19]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Eth-Benzene,MP-Xylene,RH,WS,WD,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,0.992085,0.838528,1.934549,0.063517,-0.134019,1.088182,1.201267,1.285052,1.700281,-1.565646,0.078793,-2.685253,1.263058,-0.697551,0.851835,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,1.481665,1.444147,0.927838,-0.037198,-0.134019,0.616943,1.196036,1.957611,1.731885,-1.213865,0.078793,0.145209,1.404438,-0.813583,1.261277,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,2.703168,2.044885,-0.071471,0.664182,-0.134019,0.072371,1.211147,-0.095464,2.395567,-1.143508,0.078793,-1.034150,1.632573,-0.755567,1.052175,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,2.568533,1.943100,-0.071471,1.548847,-0.134019,0.072371,1.228583,-0.095464,1.563330,-0.862084,0.078793,-2.921125,1.631770,0.622312,-1.317596,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,1.458002,1.327821,2.965934,1.350137,-0.134019,1.294604,1.273335,-0.095464,1.540756,0.052547,0.078793,0.145209,1.018857,-0.102888,-0.903996,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,-0.071324,2.936021,-0.671056,1.293882,1.156679,-0.208743,-1.147913,-0.095464,-1.419481,-2.832058,-0.812158,-1.034150,-0.502580,-1.060151,1.262762,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,-0.071324,2.584655,-1.754257,1.687671,1.000618,-0.402942,-1.060735,-0.095464,-1.487203,-2.761701,-0.812158,-1.270022,-0.296937,-1.495270,1.240188,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,-0.071324,2.377451,-1.092986,0.956348,0.735827,-0.424671,-1.038649,2.736364,-1.035718,0.052547,-0.633967,-0.798279,-0.350758,-1.248703,1.431172,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,-0.071324,1.988903,-1.043637,1.008974,0.781878,-1.502952,-1.074102,2.276192,-0.766332,-2.761701,-0.901253,-1.034150,-0.443137,-1.002135,1.459240,0.0,0.0


In [20]:
df.to_excel('DTU2025.xlsx', index=False)